# 🚗 02. YOLO11 Detection + ByteTrack + TrajectoryManager

**Tuần 2 (08/09 – 14/09)** — Kaggle Notebook

**Mục tiêu:** Detection + Tracking chạy được trên video mẫu và **hỗ trợ chạy hàng loạt (Batch Processing) trên dataset nhiều video**, output ra trajectory chuẩn hóa sẵn sàng cho Tuần 3.

## Pipeline tuần này
```
Video dataset đầu vào (1 hoặc nhiều video)
    ↓
[YOLO11s pretrained COCO] — detect: car, motorcycle, bus, truck, person, bicycle
    ↓
[ByteTrack] — gắn track ID nhất quán theo thời gian
    ↓
[TrajectoryManager] — lưu quỹ đạo, tính velocity / acceleration
    ↓
trajectories/{video_stem}_trajectories.json + tracking_manifest.json  ← INPUT CHO TUẦN 3
```

---
**GPU quota tuần này:** ~5h Kaggle (detection/tracking không nặng)

| Cell | Nội dung | Cần GPU? |
|------|----------|----------|
| 1–2  | Setup môi trường | Không |
| 3–5  | Cấu hình video & Detection YOLO11s | ✅ Có |
| 6–8  | Tracking ByteTrack trên 1 video mẫu | ✅ Có |
| 9–11 | TrajectoryManager & Phân tích chuyển động | Không |
| 12   | Visualize trajectory trên video mẫu | Không |
| 13   | Export kết quả video mẫu | Không |
| 14   | 🚀 **Batch Tracking trên toàn bộ Dataset (Nhiều Video)** | ✅ Có |
| 15   | 📊 **Kiểm tra dữ liệu & Kết nối Tuần 3 (Baseline Rules)** | Không |
| 16   | Upload kết quả về Google Drive / Kaggle Dataset | Không |


## ⚙️ Cell 1 — Setup môi trường & Clone Repository

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import json, time, random
from collections import defaultdict
import numpy as np

WORKSPACE = Path("/kaggle/working")
REPO_URL = "https://github.com/ThanhND2005/Traffic-Accident-Detection.git"
REPO_DIR = WORKSPACE / "Traffic-Accident-Detection"

# Dùng subprocess thay vì !git để không bao giờ bị lỗi IndentationError
if not REPO_DIR.exists():
    print(f"🔄 Đang clone repo từ {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("🔄 Repo đã có sẵn. Đang pull cập nhật mới nhất...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

# Chuyển working dir vào thư mục dự án
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"✅ Thư mục làm việc hiện tại: {os.getcwd()}")

# ── Phát hiện môi trường ──────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'google.colab' in sys.modules
print(f'Environment: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"}')

# ── Thư mục làm việc ─────────────────────────────────────────────────────
if IS_KAGGLE:
    WORK_DIR   = '/kaggle/working/week2_output'
    # Dataset input (nếu đã upload video lên Kaggle Dataset)
    INPUT_DIR  = '/kaggle/input'
elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/accident_detection'
    WORK_DIR   = f'{DRIVE_BASE}/results/week2_output'
    INPUT_DIR  = f'{DRIVE_BASE}/datasets'
else:
    WORK_DIR  = 'results/week2_output'
    INPUT_DIR = 'data'

os.makedirs(WORK_DIR, exist_ok=True)
print(f'✅ Work dir: {WORK_DIR}')

## ⚙️ Cell 2 — Cài thư viện + kiểm tra GPU

In [ ]:
# Cài ultralytics (YOLO11 + ByteTrack tích hợp sẵn)
!pip install -q ultralytics

import torch
from ultralytics import YOLO
import ultralytics

print(f'Ultralytics : {ultralytics.__version__}')
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA avail  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU         : {gpu.name}')
    print(f'VRAM        : {gpu.total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Không tìm thấy GPU — hãy bật Accelerator trong Settings!')

## 🎯 Cell 3 — Cấu hình đường dẫn video

> **Cách lấy video mẫu:**
> - Trên Kaggle: upload video lên dataset riêng → mount vào `/kaggle/input/`
> - Trên Colab: đặt video vào `MyDrive/accident_detection/results/sample_videos/`
> - Hoặc tải nhanh clip mẫu từ CADP/CCD dataset đã có

In [ ]:
# ── CẤU HÌNH THƯ MỤC HOẶC FILE VIDEO ──────────────────────────────────────
from pathlib import Path
import os, cv2

VIDEO_EXTS = {'.mp4', '.avi', '.mkv', '.mov'}

if IS_KAGGLE:
    # Tự động tìm kiếm thư mục chứa video trên Kaggle
    candidates = [
        Path('/kaggle/working/Traffic-Accident-Detection/datasets/cadp/videos'),
        Path('/kaggle/working/Traffic-Accident-Detection/demo/sample_videos'),
        Path('/kaggle/working/cadp_videos'),
        Path('/kaggle/input'),
    ]
    VIDEO_DIR = None
    for cand in candidates:
        if cand.exists() and any(f.suffix.lower() in VIDEO_EXTS for f in cand.rglob('*')):
            VIDEO_DIR = cand
            break
    if VIDEO_DIR is None:
        VIDEO_DIR = Path('/kaggle/working/Traffic-Accident-Detection/datasets/cadp/videos')
elif IS_COLAB:
    colab_candidates = [
        Path(f'{DRIVE_BASE}/datasets/cadp/videos'),
        Path(f'{DRIVE_BASE}/results/sample_videos'),
        Path(f'{DRIVE_BASE}/datasets'),
        REPO_DIR / 'demo/sample_videos',
    ]
    VIDEO_DIR = Path(f'{DRIVE_BASE}/datasets/cadp/videos')
    for cand in colab_candidates:
        if cand.exists() and any(f.suffix.lower() in VIDEO_EXTS for f in cand.rglob('*')):
            VIDEO_DIR = cand
            break
else:
    local_candidates = [
        Path('datasets/cadp/videos'),
        Path('demo/sample_videos'),
        Path('data'),
        Path('datasets'),
    ]
    VIDEO_DIR = Path('datasets/cadp/videos')
    for cand in local_candidates:
        if cand.exists() and any(f.suffix.lower() in VIDEO_EXTS for f in cand.rglob('*')):
            VIDEO_DIR = cand
            break

# Hoặc bạn có thể tự gán trực tiếp đường dẫn thư mục/file của bạn tại đây:
# VIDEO_DIR = Path('/kaggle/input/cadp-traffic-videos')
# ──────────────────────────────────────────────────────────────────────────

VIDEO_PATHS = []
if VIDEO_DIR.is_file():
    VIDEO_PATHS = [VIDEO_DIR]
elif VIDEO_DIR.is_dir():
    VIDEO_PATHS = sorted([f for f in VIDEO_DIR.rglob('*') if f.suffix.lower() in VIDEO_EXTS])

if not VIDEO_PATHS:
    print(f'⚠️  Không tìm thấy video nào trong: {VIDEO_DIR}')
    print('   → Hãy kiểm tra lại đường dẫn VIDEO_DIR hoặc tải video vào thư mục này.')
    VIDEO_PATH = None
else:
    print(f'🎉 Đã tìm thấy TỔNG CỘNG: {len(VIDEO_PATHS)} video trong thư mục: {VIDEO_DIR}\n')
    print(f'{"#":<4}{"Tên Video":<38}{"Dung lượng":<12}{"Độ phân giải":<14}{"FPS":<8}{"Thời lượng"}')
    print('-' * 88)

    # Hiển thị thông số 5 video đầu tiên làm mẫu
    for i, vpath in enumerate(VIDEO_PATHS[:5]):
        size_mb = vpath.stat().st_size / (1024 * 1024)
        cap = cv2.VideoCapture(str(vpath))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = frames / fps if fps > 0 else 0
        cap.release()
        print(f'[{i}]  {vpath.name:<38}{size_mb:>7.1f} MB  {f"{w}x{h}":<14}{fps:>5.1f}   {duration:>6.1f}s ({frames} frames)')

    if len(VIDEO_PATHS) > 5:
        print(f'... và {len(VIDEO_PATHS) - 5} video khác.')

    # ── CHỌN 1 VIDEO MẪU ĐỂ CHẠY CÁC CELL TIẾP THEO (Cell 5, 7, 12) ────────
    SAMPLE_INDEX = 0  # Đổi số này (0, 1, 2,...) để chọn video khác
    VIDEO_PATH = str(VIDEO_PATHS[SAMPLE_INDEX])
    print(f'\n👉 Đã chọn video mẫu: [{SAMPLE_INDEX}] {os.path.basename(VIDEO_PATH)}')
    print(f'   (Đường dẫn: {VIDEO_PATH})')


## 🔍 Cell 4 — Load YOLO11s pretrained (COCO)

**Tại sao YOLO11s?**
| Model | Params | mAP COCO | Speed | Dùng khi |
|-------|--------|----------|-------|----------|
| yolo11n | 2.6M | 39.5 | Rất nhanh | Edge, demo nhanh |
| **yolo11s** | **9.4M** | **47.0** | **Nhanh** | **✅ Tuần 2-4** |
| yolo11m | 20.1M | 51.5 | TB | Cần accuracy cao |

→ yolo11s: cân bằng tốt nhất, đủ tốt cho prototype, inference nhanh trên T4.

In [ ]:
# Tải yolo11s.pt (tự động download từ Ultralytics nếu chưa có)
model = YOLO('yolo11s.pt')

# Class IDs cần giữ (COCO pretrained)
VEHICLE_CLASSES = [0, 1, 2, 3, 5, 7]  # person, bicycle, car, motorcycle, bus, truck
CLASS_NAMES = {
    0: 'person', 1: 'bicycle', 2: 'car',
    3: 'motorcycle', 5: 'bus', 7: 'truck'
}

print('✅ YOLO11s loaded!')
print(f'   Classes kept: {CLASS_NAMES}')

## 🔍 Cell 5 — Test detection trên vài frame mẫu

In [ ]:
import cv2
import matplotlib.pyplot as plt
from collections import Counter

# Chạy detection trên tối đa 5 giây đầu video để test nhanh
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
sample_frames = int(fps * 5)  # 5 giây đầu

print(f'🔍 Testing detection on first {sample_frames} frames...')
all_class_counts = Counter()
preview_frames = []

for f_idx in range(sample_frames):
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(
        source=frame,
        conf=0.4,
        classes=VEHICLE_CLASSES,
        verbose=False
    )

    r = results[0]
    if r.boxes is not None:
        for cls in r.boxes.cls.cpu().numpy().astype(int):
            all_class_counts[CLASS_NAMES.get(cls, str(cls))] += 1

    # Lưu 3 frame để preview
    if f_idx in [0, sample_frames // 2, sample_frames - 1]:
        annotated = r.plot()
        preview_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))

cap.release()

# Hiển thị kết quả
print(f'\n📊 Phân bố class trong {sample_frames} frames đầu:')
for cls, cnt in all_class_counts.most_common():
    print(f'   {cls:12s}: {cnt:5d} detections')

# Plot preview
if preview_frames:
    fig, axes = plt.subplots(1, len(preview_frames), figsize=(18, 5))
    if len(preview_frames) == 1:
        axes = [axes]
    for ax, img in zip(axes, preview_frames):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle('YOLO11s Detection Preview (frame 0, middle, end)', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/detection_preview.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'\n✅ Preview saved: {WORK_DIR}/detection_preview.png')

## 📡 Cell 6 — Cấu hình ByteTrack

**Tham số tùy chỉnh cho giao thông Việt Nam:**

| Tham số | Mặc định | Tùy chỉnh | Lý do |
|---------|----------|-----------|-------|
| `track_high_thresh` | 0.5 | **0.4** | Xe máy nhỏ thường có conf thấp |
| `new_track_thresh` | 0.6 | **0.5** | Khởi tạo track sớm hơn |
| `track_buffer` | 30 | **60** | Giữ track lâu hơn khi bị che khuất |
| `track_low_thresh` | 0.1 | 0.1 | Giữ mặc định |
| `match_thresh` | 0.8 | 0.8 | Giữ mặc định |

In [ ]:
import yaml

# Tạo file ByteTrack config tùy chỉnh
bytetrack_cfg = {
    'tracker_type'     : 'bytetrack',
    'track_high_thresh': 0.4,   # hạ từ 0.5 cho xe máy nhỏ
    'track_low_thresh' : 0.1,
    'new_track_thresh' : 0.5,   # hạ từ 0.6
    'track_buffer'     : 60,    # giữ track 60 frame khi mất (~2s ở 30fps)
    'match_thresh'     : 0.8,
    'fuse_score'       : True,
}

tracker_yaml = f'{WORK_DIR}/bytetrack_custom.yaml'
with open(tracker_yaml, 'w') as f:
    yaml.dump(bytetrack_cfg, f, default_flow_style=False)

print(f'✅ ByteTrack config saved: {tracker_yaml}')
print(yaml.dump(bytetrack_cfg, default_flow_style=False))

## 📡 Cell 7 — Chạy YOLO11 + ByteTrack tracking

> ⏱️ **Thời gian ước tính:** ~1 phút cho video 30s trên T4 GPU.

In [ ]:
import time

# Thu thập raw tracking data (frame_id → list of tracks)
raw_tracking = []   # list of {frame_id, tracks: [{track_id, bbox, class_id, confidence}]}

t0 = time.time()
print('▶️  Running YOLO11s + ByteTrack...')

# stream=True: xử lý từng frame, tiết kiệm RAM (quan trọng với video dài)
results_gen = model.track(
    source=VIDEO_PATH,
    persist=True,           # giữ track ID liên tục giữa các frame
    tracker=tracker_yaml,   # dùng config tùy chỉnh
    conf=0.4,
    iou=0.5,
    classes=VEHICLE_CLASSES,
    imgsz=640,
    stream=True,            # ← quan trọng: không load toàn bộ vào RAM
    verbose=False,
)

for frame_idx, result in enumerate(results_gen):
    frame_data = {'frame_id': frame_idx, 'tracks': []}

    if result.boxes is not None and result.boxes.id is not None:
        tids   = result.boxes.id.cpu().numpy().astype(int)
        bboxes = result.boxes.xyxy.cpu().numpy()
        clss   = result.boxes.cls.cpu().numpy().astype(int)
        cfs    = result.boxes.conf.cpu().numpy()

        for tid, bbox, cls, cf in zip(tids, bboxes, clss, cfs):
            frame_data['tracks'].append({
                'track_id'  : int(tid),
                'bbox'      : bbox.tolist(),
                'class_id'  : int(cls),
                'confidence': round(float(cf), 4),
            })

    raw_tracking.append(frame_data)

    if frame_idx % 150 == 0 and frame_idx > 0:
        elapsed = time.time() - t0
        active  = len(frame_data['tracks'])
        print(f'  Frame {frame_idx:5d}  active_tracks={active:3d}  elapsed={elapsed:.1f}s')

elapsed = time.time() - t0
total_frames = len(raw_tracking)
print(f'\n✅ Tracking done!')
print(f'   Frames processed: {total_frames}')
print(f'   Total time      : {elapsed:.1f}s  ({total_frames/elapsed:.1f} FPS effective)')
print(f'   Unique track IDs: {len({t["track_id"] for f in raw_tracking for t in f["tracks"]})}')

## 📡 Cell 8 — Lưu raw tracking data

In [ ]:
import json

raw_json_path = f'{WORK_DIR}/tracking_raw.json'
with open(raw_json_path, 'w') as f:
    json.dump(raw_tracking, f)

size_kb = os.path.getsize(raw_json_path) / 1024
print(f'✅ Raw tracking saved: {raw_json_path}  ({size_kb:.0f} KB)')

# Preview frame 0
print(f'\n📋 Sample (frame 0):')
print(json.dumps(raw_tracking[0], indent=2))

## 🗃️ Cell 9 — TrajectoryManager: build từ raw tracking

In [ ]:
# Import TrajectoryManager từ src
# Trên Kaggle: cần clone repo trước hoặc copy code vào đây
# ── Option A: import từ repo ─────────────────────────────────────────────
# !git clone https://github.com/YOUR_REPO/accident-detection.git repo
# sys.path.insert(0, 'repo')
# from src.tracking.trajectory_manager import TrajectoryManager

# ── Option B: inline class (paste code từ src/tracking/trajectory_manager.py)
# Xem file: https://github.com/YOUR_REPO/accident-detection/blob/main/src/tracking/trajectory_manager.py

# ── Dùng tạm class inline để notebook tự chứa ───────────────────────────
from collections import deque
from typing import Dict, List, Tuple, Optional, Any
import numpy as np, json, csv, os

COCO_VEHICLE_CLASSES = {
    0:'person', 1:'bicycle', 2:'car', 3:'motorcycle', 5:'bus', 7:'truck'
}

# [Class TrajectoryManager đầy đủ — xem src/tracking/trajectory_manager.py]
# Để notebook ngắn gọn, ta import trực tiếp:
try:
    sys.path.insert(0, str(REPO_DIR) if 'REPO_DIR' in locals() else ('/kaggle/working/Traffic-Accident-Detection' if IS_KAGGLE else '.'))
    from src.tracking.trajectory_manager import TrajectoryManager
    print('✅ TrajectoryManager imported from src/')
except ImportError:
    print('⚠️  Không import được src/ — hãy clone repo hoặc paste class inline.')
    print('   Xem: src/tracking/trajectory_manager.py trong repo.')

## 🗃️ Cell 10 — Populate TrajectoryManager

In [ ]:
# Tăng max_history lên 300 (~10s ở 30fps) để lưu đủ dài cho việc phân tích
manager = TrajectoryManager(max_history=300)

for frame_data in raw_tracking:
    fid    = frame_data['frame_id']
    tracks = frame_data['tracks']

    if tracks:
        tids   = np.array([t['track_id']   for t in tracks], dtype=np.int32)
        bboxes = np.array([t['bbox']        for t in tracks], dtype=np.float32)
        clss   = np.array([t['class_id']    for t in tracks], dtype=np.int32)
        cfs    = np.array([t['confidence']  for t in tracks], dtype=np.float32)
        manager.update(fid, tids, bboxes, clss, cfs)

    # Cleanup định kỳ: Chỉ dùng cho streaming camera 24/7 để tiết kiệm RAM.
    # Với video offline, comment để giữ toàn bộ quỹ đạo cho Cell 11, 12, 13 xuất CSV/JSON.
    # if fid > 0 and fid % 300 == 0:
    #     manager.cleanup_old_tracks(fid, max_age=90)

# Thống kê
summary = manager.get_summary()
print('📊 TrajectoryManager Summary:')
for k, v in summary.items():
    print(f'   {k:25s}: {v}')


## 🗃️ Cell 11 — Phân tích chuyển động (velocity, acceleration)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Lấy danh sách track để phân tích toàn bộ video
# Ưu tiên các track dài (>= 20 frames), tự động fallback nếu video ngắn hoặc track phân mảnh
min_len = 20
if hasattr(manager, 'get_all_tracks'):
    candidate_tracks = manager.get_all_tracks(min_length=min_len)
elif hasattr(manager, 'get_active_tracks'):
    try:
        candidate_tracks = manager.get_active_tracks(min_length=min_len, max_age=None)
    except TypeError:
        candidate_tracks = [tid for tid, hist in manager.tracks.items() if len(hist) >= min_len]
else:
    candidate_tracks = [tid for tid, hist in manager.tracks.items() if len(hist) >= min_len]

# Fallback tự động: nếu không có track >= 20 frames, thử các ngưỡng nhỏ hơn
if not candidate_tracks:
    min_len = 10
    candidate_tracks = [tid for tid, hist in manager.tracks.items() if len(hist) >= min_len]
if not candidate_tracks:
    min_len = 5
    candidate_tracks = [tid for tid, hist in manager.tracks.items() if len(hist) >= min_len]
if not candidate_tracks:
    min_len = 2  # Tối thiểu 2 frames để tính được vận tốc (diff)
    candidate_tracks = [tid for tid, hist in manager.tracks.items() if len(hist) >= min_len]

print(f'✅ Tracks selected for analysis (≥{min_len} frames): {len(candidate_tracks)}')

cols = ['track_id', 'class', 'length_frames', 'avg_speed_px_f', 'max_speed_px_f', 'max_decel_px_f2', 'max_dir_change_deg']

rows = []
for tid in candidate_tracks:
    speed  = manager.get_speed(tid)
    accel  = manager.get_acceleration(tid)
    dtheta = manager.get_direction_changes(tid)
    last   = manager.get_last_state(tid)

    if speed is not None and len(speed) > 0:
        rows.append({
            'track_id'         : tid,
            'class'            : COCO_VEHICLE_CLASSES.get(last['class_id'], '?') if last else '?',
            'length_frames'    : len(manager.tracks[tid]),
            'avg_speed_px_f'   : round(float(np.mean(speed)), 2),
            'max_speed_px_f'   : round(float(np.max(speed)), 2),
            'max_decel_px_f2'  : round(float(np.min(np.diff(speed))), 2) if len(speed) > 1 else 0.0,
            'max_dir_change_deg': round(float(np.max(np.abs(dtheta))), 1) if dtheta is not None and len(dtheta) > 0 else 0.0,
        })

# Khởi tạo DataFrame có danh sách cột rõ ràng để tránh KeyError khi rows rỗng
df = pd.DataFrame(rows, columns=cols)

if not df.empty:
    df = df.sort_values('avg_speed_px_f', ascending=False)
    print('\n📈 Track Analysis (top 15):')
    print(df.head(15).to_string(index=False))
else:
    print('\n⚠️ Chưa có track nào đủ dữ liệu để phân tích tốc độ (cần tối thiểu 2 frames/track).')

# Plot speed profiles
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if not df.empty and len(df) > 0:
    # Speed histogram
    num_bins = min(20, max(1, len(df)))
    axes[0].hist(df['avg_speed_px_f'], bins=num_bins, color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Avg Speed (pixel/frame)')
    axes[0].set_ylabel('Number of Tracks')
    axes[0].set_title(f'Speed Distribution of All Tracks (N={len(df)})')

    # Theo dõi tốc độ 1 track cụ thể (lấy track có avg_speed cao nhất)
    sample_tid = int(df.iloc[0]['track_id'])
    speed_arr  = manager.get_speed(sample_tid)
    if speed_arr is not None and len(speed_arr) > 0:
        axes[1].plot(speed_arr, color='tomato', label=f'Track #{sample_tid}')
        axes[1].set_xlabel('Frame (relative)')
        axes[1].set_ylabel('Speed (px/frame)')
        axes[1].set_title(f'Speed Profile — Track #{sample_tid}')
        axes[1].axhline(np.mean(speed_arr), color='gray', ls='--', label=f'mean: {np.mean(speed_arr):.1f}')
        axes[1].legend()
else:
    axes[0].text(0.5, 0.5, 'No track data available', ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('Speed Distribution of All Tracks')
    axes[1].text(0.5, 0.5, 'No track data available', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Speed Profile')

plt.tight_layout()
save_dir = WORK_DIR if 'WORK_DIR' in locals() else '.'
os.makedirs(save_dir, exist_ok=True)
speed_analysis_path = os.path.join(save_dir, 'speed_analysis.png')
plt.savefig(speed_analysis_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'\n✅ Speed analysis saved: {speed_analysis_path}')


## 🎬 Cell 12 — Visualize trajectory trên video

> ⏱️ **Thời gian ước tính:** ~30–60 giây cho video 30s (CPU, không cần GPU).

In [ ]:
import cv2, random, time
from collections import defaultdict

TAIL_FRAMES = 30   # số frame lịch sử vẽ đuôi trajectory

def get_color(track_id):
    rng = random.Random(track_id * 2654435761)
    return (rng.randint(80,255), rng.randint(80,255), rng.randint(80,255))  # BGR

# Tổ chức lại history theo frame_id để lookup nhanh
frame_lookup = defaultdict(list)
for frame_data in raw_tracking:
    for t in frame_data['tracks']:
        frame_lookup[frame_data['frame_id']].append(t)

# Build center history cho mỗi track
center_history = {}   # tid → [(frame_id, (cx, cy))]
for tid, history in manager.tracks.items():
    center_history[tid] = [
        (s['frame_id'], (s['center'][0], s['center'][1]))
        for s in history
    ]

# Video writer
cap = cv2.VideoCapture(VIDEO_PATH)
fps_vid = cap.get(cv2.CAP_PROP_FPS) or 30
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out_path = f'{WORK_DIR}/trajectory_visualization.mp4'
writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps_vid, (W, H))

print(f'🎬 Rendering trajectory video...')
t0 = time.time()

for frame_idx in range(total):
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Vẽ đuôi trajectory (mờ dần về quá khứ)
    for tid, hist in center_history.items():
        tail = [(f,c) for f,c in hist if frame_idx - TAIL_FRAMES <= f <= frame_idx]
        color = get_color(tid)
        for i in range(1, len(tail)):
            alpha = i / len(tail)
            c = tuple(int(v * alpha) for v in color)
            p1 = (int(tail[i-1][1][0]), int(tail[i-1][1][1]))
            p2 = (int(tail[i][1][0]),   int(tail[i][1][1]))
            cv2.line(frame, p1, p2, c, 2, cv2.LINE_AA)
        if tail:
            cur = (int(tail[-1][1][0]), int(tail[-1][1][1]))
            cv2.circle(frame, cur, 4, color, -1)

    # 2. Vẽ bbox + label cho frame hiện tại
    for t in frame_lookup.get(frame_idx, []):
        tid, bbox = t['track_id'], t['bbox']
        color = get_color(tid)
        x1,y1,x2,y2 = map(int, bbox)
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        cls_name = CLASS_NAMES.get(t['class_id'], str(t['class_id']))
        label = f"{cls_name} #{tid}"
        cv2.putText(frame, label, (x1, max(y1-6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

    # 3. Overlay info
    n_active = len(frame_lookup.get(frame_idx, []))
    cv2.putText(frame, f'Frame: {frame_idx}  Tracks: {n_active}',
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    writer.write(frame)

cap.release()
writer.release()
print(f'✅ Video saved: {out_path}  ({time.time()-t0:.1f}s)')

## 💾 Cell 13 — Export trajectories.csv + trajectories.json

In [ ]:
# Export CSV
csv_path = f'{WORK_DIR}/trajectories.csv'
manager.export_csv(csv_path)

# Export JSON
json_path = f'{WORK_DIR}/trajectories.json'
manager.export_json(json_path)

# Ghi thêm metadata video_path vào JSON để Tuần 3 tự động nhận diện video gốc
try:
    with open(json_path, 'r', encoding='utf-8') as f:
        tdata = json.load(f)
    if 'meta' not in tdata:
        tdata['meta'] = {}
    tdata['meta']['video_path'] = str(VIDEO_PATH) if VIDEO_PATH else ''
    tdata['meta']['video_name'] = os.path.basename(VIDEO_PATH) if VIDEO_PATH else ''
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(tdata, f, indent=2, ensure_ascii=False)
except Exception as e:
    print(f'ℹ️ Lưu ý metadata: {e}')

# Đọc lại CSV preview
import pandas as pd
df_full = pd.read_csv(csv_path)
print(f'\n📋 trajectories.csv: {len(df_full)} rows × {len(df_full.columns)} cols')
print(df_full.head(8).to_string(index=False))

print(f'\n📁 Files ready for Week 3:')
for path in [csv_path, json_path, raw_json_path]:
    size = os.path.getsize(path) / 1024
    print(f'   {os.path.basename(path):35s}  {size:.0f} KB')


## 🚀 Cell 14 — Batch Tracking trên toàn bộ Dataset (Nhiều Video)

> **Mục tiêu:** Chạy tự động YOLO11s + ByteTrack trên toàn bộ danh sách `VIDEO_PATHS` để tạo bộ dữ liệu trajectory cho **Tuần 3** (Rule-Based Baseline & Evaluation).
>
> **Đặc điểm thiết kế:**
> - Duyệt qua từng video trong danh sách `VIDEO_PATHS`.
> - Mỗi video khởi tạo 1 instance `TrajectoryManager` riêng biệt (tránh trùng lặp `track_id` giữa các video).
> - Xuất file kết quả theo từng video: `trajectories/{video_stem}_trajectories.json` và `trajectories/{video_stem}_trajectories.csv`.
> - Hỗ trợ `SKIP_EXISTING = True` để tự động resume nếu notebook bị ngắt kết nối (tiết kiệm quota GPU).
> - Tạo file `tracking_manifest.json` và `batch_summary.csv` tổng hợp toàn bộ dataset để Tuần 3 nạp tự động.


In [ ]:
import os, time, json
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# ── CẤU HÌNH BATCH PROCESSING ─────────────────────────────────────────────
MAX_VIDEOS = None          # None: Chạy TẤT CẢ video tìm thấy; hoặc đặt số nguyên (vd: 10, 50) để chạy thử nghiệm
SKIP_EXISTING = True       # True: Bỏ qua video đã xử lý trước đó (hỗ trợ resume khi session ngắt)
SAVE_CSV_PER_VIDEO = True  # True: Xuất kèm file .csv riêng cho từng video
MAX_HISTORY = 300          # Số frame lịch sử tối đa cho mỗi track (~10s ở 30fps)

# Thư mục lưu kết quả trajectory cho Tuần 3
TRAJ_DIR = os.path.join(WORK_DIR, 'trajectories')
os.makedirs(TRAJ_DIR, exist_ok=True)

target_videos = VIDEO_PATHS[:MAX_VIDEOS] if MAX_VIDEOS else VIDEO_PATHS
print(f'🚀 Bắt đầu Batch Tracking trên {len(target_videos)} video...')
print(f'📂 Thư mục output trajectory: {TRAJ_DIR}\n')

batch_records = []
total_t0 = time.time()

for v_idx, vpath in enumerate(target_videos):
    vpath = Path(vpath)
    stem = vpath.stem
    json_out = os.path.join(TRAJ_DIR, f'{stem}_trajectories.json')
    csv_out = os.path.join(TRAJ_DIR, f'{stem}_trajectories.csv')

    # 1. Kiểm tra nếu đã có kết quả và bật SKIP_EXISTING
    if SKIP_EXISTING and os.path.exists(json_out):
        print(f'[{v_idx+1:3d}/{len(target_videos)}] ⏩ Đã tồn tại, bỏ qua: {vpath.name}')
        try:
            with open(json_out, 'r', encoding='utf-8') as f:
                existing_data = json.load(f)
            meta = existing_data.get('meta', {})
            batch_records.append({
                'video_name': vpath.name,
                'video_stem': stem,
                'video_path': str(vpath),
                'total_frames': meta.get('last_frame', 0) + 1,
                'total_tracks': meta.get('total_tracks', 0),
                'fps': 30.0,
                'duration_sec': 0.0,
                'elapsed_sec': 0.0,
                'trajectory_json': json_out,
                'trajectory_csv': csv_out if os.path.exists(csv_out) else None,
                'status': 'skipped_existing',
            })
        except Exception:
            pass
        continue

    t_vid0 = time.time()
    cap = cv2.VideoCapture(str(vpath))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print(f'[{v_idx+1:3d}/{len(target_videos)}] ▶️ Đang tracking: {vpath.name} ({W}x{H}, {fps:.1f} FPS, {num_frames} frames)...')

    # 2. Khởi tạo TrajectoryManager mới cho từng video (tránh va chạm track_id)
    vid_manager = TrajectoryManager(max_history=MAX_HISTORY)

    # 3. Chạy tracking với stream=True để tối ưu RAM
    results_gen = model.track(
        source=str(vpath),
        persist=True,
        tracker=tracker_yaml,
        conf=0.4,
        iou=0.5,
        classes=VEHICLE_CLASSES,
        imgsz=640,
        stream=True,
        verbose=False,
    )

    frame_idx = 0
    for result in results_gen:
        if result.boxes is not None and result.boxes.id is not None:
            tids = result.boxes.id.cpu().numpy().astype(np.int32)
            bboxes = result.boxes.xyxy.cpu().numpy()
            clss = result.boxes.cls.cpu().numpy().astype(np.int32)
            cfs = result.boxes.conf.cpu().numpy()
            vid_manager.update(frame_idx, tids, bboxes, clss, cfs)
        frame_idx += 1

    # 4. Xuất file kết quả theo từng video
    vid_manager.export_json(json_out)
    if SAVE_CSV_PER_VIDEO:
        vid_manager.export_csv(csv_out)

    try:
        with open(json_out, 'r', encoding='utf-8') as f:
            tdata = json.load(f)
        if 'meta' not in tdata:
            tdata['meta'] = {}
        tdata['meta']['video_path'] = str(vpath)
        tdata['meta']['video_name'] = vpath.name
        with open(json_out, 'w', encoding='utf-8') as f:
            json.dump(tdata, f, indent=2, ensure_ascii=False)
    except Exception:
        pass

    elapsed_vid = time.time() - t_vid0
    summary = vid_manager.get_summary()

    record = {
        'video_name': vpath.name,
        'video_stem': stem,
        'video_path': str(vpath),
        'total_frames': frame_idx,
        'total_tracks': summary.get('total_tracks', 0),
        'fps': round(float(fps), 2),
        'duration_sec': round(float(frame_idx / fps), 1) if fps > 0 else 0,
        'elapsed_sec': round(float(elapsed_vid), 1),
        'trajectory_json': json_out,
        'trajectory_csv': csv_out if SAVE_CSV_PER_VIDEO else None,
        'status': 'success',
    }
    batch_records.append(record)
    print(f'      ✅ Xong trong {elapsed_vid:.1f}s | {summary.get("total_tracks", 0)} tracks | {frame_idx} frames')

# ── 5. LƯU MANIFEST & SUMMARY TỔNG HỢP CHO TUẦN 3 ────────────────────────
manifest_path = os.path.join(WORK_DIR, 'tracking_manifest.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(batch_records, f, indent=2, ensure_ascii=False)

summary_csv_path = os.path.join(WORK_DIR, 'batch_summary.csv')
df_batch = pd.DataFrame(batch_records)
df_batch.to_csv(summary_csv_path, index=False)

total_time = time.time() - total_t0
print('\n' + '='*70)
print(f'🎉 HOÀN THÀNH BATCH TRACKING {len(batch_records)} VIDEO!')
print(f'⏱️ Tổng thời gian: {total_time:.1f}s ({total_time/60:.1f} phút)')
print(f'📋 Tracking Manifest JSON: {manifest_path}')
print(f'📊 Batch Summary CSV     : {summary_csv_path}')
print(f'📂 Thư mục Trajectories  : {TRAJ_DIR}')
print('='*70)


## 📊 Cell 15 — Kiểm tra dữ liệu & Kết nối với Tuần 3 (Rule-Based Baseline)

> **Mục tiêu:** Kiểm tra tính toàn vẹn của dữ liệu xuất ra và minh họa cách **Tuần 3** (`03_baseline_rules.ipynb`) sẽ nạp các file trajectory này để phát hiện va chạm.


In [ ]:
import json
import pandas as pd
import numpy as np

# 1. Hiển thị bảng tổng kết Batch Processing
if os.path.exists(summary_csv_path):
    df_sum = pd.read_csv(summary_csv_path)
    print(f'📊 Tổng số video trong dataset : {len(df_sum)}')
    print(f'🚗 Tổng số tracks ghi nhận     : {df_sum["total_tracks"].sum()}')
    print(f'⏱️ Tổng thời lượng video       : {df_sum["duration_sec"].sum():.1f}s')
    print('\n📋 Bảng tổng hợp các video:')
    cols_show = [c for c in ['video_name', 'total_frames', 'total_tracks', 'elapsed_sec', 'status'] if c in df_sum.columns]
    print(df_sum[cols_show].head(15).to_string(index=False))

# 2. Minh họa cách Tuần 3 nạp dữ liệu từ tracking_manifest.json
print('\n' + '='*70)
print('🔗 MINH HỌA: CÁCH TUẦN 3 SẼ ĐỌC DỮ LIỆU TỪ TUẦN 2')
print('='*70)

manifest_file = os.path.join(WORK_DIR, 'tracking_manifest.json')
with open(manifest_file, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

if manifest:
    sample_item = manifest[0]
    sample_json = sample_item['trajectory_json']
    print(f'👉 Nạp thử video: {sample_item["video_name"]}')
    print(f'   Đường dẫn JSON: {sample_json}')
    with open(sample_json, 'r', encoding='utf-8') as f:
        traj_data = json.load(f)

    print(f'   Meta: {traj_data.get("meta", {})}')
    tracks_dict = traj_data.get('tracks', {})
    print(f'   Số lượng tracks trong file: {len(tracks_dict)}')

    if tracks_dict:
        first_tid = list(tracks_dict.keys())[0]
        pts = tracks_dict[first_tid]
        print(f'   Track #{first_tid} ({pts[0].get("class_name", "?")}): {len(pts)} frames')
        sample_centers = np.array([p['center'] for p in pts], dtype=np.float32)
        sample_bboxes = np.array([p['bbox'] for p in pts], dtype=np.float32)
        print(f'   Shape centers: {sample_centers.shape} | Shape bboxes: {sample_bboxes.shape}')
        print('   ✅ Dữ liệu hoàn toàn tương thích với MotionFeatureExtractor & RuleBasedAccidentDetector của Tuần 3!')


## ☁️ Cell 16 — Upload kết quả về Google Drive

> Thực hiện sau khi đã commit notebook (Save & Run All) để lấy output.


In [ ]:
# ── CÁCH 1: Dùng pydrive2 (Colab có sẵn auth) ─────────────────────────
if IS_COLAB:
    print('Colab: file đã lưu trực tiếp vào Drive. Không cần upload thêm.')
    print(f'  ✅ {WORK_DIR}')

# ── CÁCH 2: Kaggle — tạo Output Dataset ────────────────────────────────
elif IS_KAGGLE:
    print('Kaggle: Để lưu kết quả về Drive, dùng một trong 2 cách:')
    print()
    print('Cách A (khuyến nghị) — Kaggle Output Dataset:')
    print('  1. Click "Save Version" → "Save & Run All (Commit)"')
    print('  2. Sau khi xong, vào Output tab → "+ New Dataset"')
    print('  3. Dataset sẽ được mount lại ở notebook khác: /kaggle/input/<tên>/')
    print()
    print('Cách B — Upload thủ công qua rclone:')
    print('  !curl https://rclone.org/install.sh | sudo bash -q')
    print('  !rclone copy /kaggle/working/week2_output/ gdrive:accident_detection/results/week2_output/ --progress')
    print()
    print(f'📁 Files & thư mục tại: {WORK_DIR}')
    for fn in sorted(os.listdir(WORK_DIR)):
        fpath = os.path.join(WORK_DIR, fn)
        if os.path.isdir(fpath):
            n_files = len(os.listdir(fpath))
            print(f'   📁 {fn:38s}  ({n_files} files)')
        else:
            size = os.path.getsize(fpath) / 1024
            print(f'   📄 {fn:38s}  {size:.0f} KB')


---

## ✅ Checklist Deliverables Tuần 2

| # | Item | File | Status |
|---|------|------|--------|
| 1 | YOLO11s detection chạy được | `detection_preview.png` | ☐ |
| 2 | ByteTrack tracking hoạt động | `tracking_raw.json` | ☐ |
| 3 | `TrajectoryManager` đầy đủ | `src/tracking/trajectory_manager.py` | ☐ |
| 4 | Phân tích speed/accel video mẫu | `speed_analysis.png` | ☐ |
| 5 | Video demo trajectory mẫu | `trajectory_visualization.mp4` | ☐ |
| 6 | Export CSV + JSON video mẫu | `trajectories.csv`, `trajectories.json` | ☐ |
| 7 | **Batch tracking dataset nhiều video** | `trajectories/*_trajectories.json` | ☐ |
| 8 | **Manifest & Summary cho Tuần 3** | `tracking_manifest.json`, `batch_summary.csv` | ☐ |
| 9 | Sync lên Google Drive / Kaggle Dataset | — | ☐ |

---

**Tuần tiếp theo:** Tuần 3 — Rule-Based Baseline sẽ đọc `tracking_manifest.json` và các file trong thư mục `trajectories/`
để phát hiện tai nạn trên toàn bộ dataset bằng các luật: giảm tốc đột ngột, thay đổi hướng, IoU chồng lấn, và tính toán metrics (Precision, Recall, F1, FAR/h).
